#### EDA2 sparksql

Nama : Beatrice Nathania
NIM : 2802490946

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.appName("Stund").getOrCreate()

In [3]:
df = spark.read.option("inferSchema", True).option("header", True).csv("student_performance_updated_1000.csv")
df.createOrReplaceTempView("student")

In [4]:
df.show(5, False)

+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+
|StudentID|Name   |Gender|AttendanceRate|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|ParentalSupport|FinalGrade|Study Hours|Attendance (%)|Online Classes Taken|
+---------+-------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+
|1.0      |John   |Male  |85.0          |15.0             |78.0         |1.0                      |High           |80.0      |4.8        |59.0          |false               |
|2.0      |Sarah  |Female|90.0          |20.0             |85.0         |2.0                      |Medium         |87.0      |2.2        |70.0          |true                |
|3.0      |Alex   |Male  |78.0          |10.0             |65.0         |0.0                      |Low            |68.0      

In [5]:
# Descriptive statistic

spark.sql("""
    SELECT 
    COUNT(*) AS total_record, 
    AVG(AttendanceRate) AS mean_attendance, 
    AVG(StudyHoursPerWeek) AS `mean study_Hours`
    FROM student
""").show()

+------------+-----------------+------------------+
|total_record|  mean_attendance|  mean study_Hours|
+------------+-----------------+------------------+
|        1000|85.51041666666667|17.630526315789474|
+------------+-----------------+------------------+



In [6]:
#cek struktur data

spark.sql("DESCRIBE Student").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|           StudentID|   double|   NULL|
|                Name|   string|   NULL|
|              Gender|   string|   NULL|
|      AttendanceRate|   double|   NULL|
|   StudyHoursPerWeek|   double|   NULL|
|       PreviousGrade|   double|   NULL|
|ExtracurricularAc...|   double|   NULL|
|     ParentalSupport|   string|   NULL|
|          FinalGrade|   double|   NULL|
|         Study Hours|   double|   NULL|
|      Attendance (%)|   double|   NULL|
|Online Classes Taken|  boolean|   NULL|
+--------------------+---------+-------+



In [7]:
# Common utility fnction
#tambahkan kolom attendace yang bukan percentage
query = """
SELECT
StudentID,
Name,
Gender,
`Attendance (%)`,
`Attendance (%)` /100 AS `Attendance (not %)`
FROM Student
"""

In [8]:
spark.sql(query).show(5)

+---------+-------+------+--------------+------------------+
|StudentID|   Name|Gender|Attendance (%)|Attendance (not %)|
+---------+-------+------+--------------+------------------+
|      1.0|   John|  Male|          59.0|              0.59|
|      2.0|  Sarah|Female|          70.0|               0.7|
|      3.0|   Alex|  Male|          92.0|              0.92|
|      4.0|Michael|  Male|          96.0|              0.96|
|      5.0|   Emma|Female|          97.0|              0.97|
+---------+-------+------+--------------+------------------+
only showing top 5 rows



In [9]:
# filter study hour yang lebih besar sama dengan 4

spark.sql(
    """
    SELECT
    Name,
    FinalGrade,
    `Study Hours`
    FROM Student
    WHERE `Study Hours` >= 4
    """
).show(10)

+--------------------+----------+-----------+
|                Name|FinalGrade|Study Hours|
+--------------------+----------+-----------+
|                John|      80.0|        4.8|
|                Alex|      68.0|        4.6|
|                Emma|      85.0|        4.1|
|              Daniel|      62.0|        4.5|
|      James Williams|      88.0|        4.4|
|         James Smith|      92.0|        4.1|
|        James Duncan|      72.0|        4.3|
|         Susan Brown|      90.0|        4.4|
|Alexandra Harrington|      87.0|        4.3|
|       Justin Martin|      85.0|        4.1|
+--------------------+----------+-----------+
only showing top 10 rows



In [10]:
#urutkan FinalGrade tertinnggi

spark.sql(
    """
    SELECT
    Name,
    StudyHoursPerWeek,
    FinalGrade
    FROM Student
    ORDER BY FinalGrade DESC
    """
).show(5)

+--------------+-----------------+----------+
|          Name|StudyHoursPerWeek|FinalGrade|
+--------------+-----------------+----------+
|   Jeremy Hall|             NULL|      92.0|
|   James Smith|             12.0|      92.0|
|       Michael|             25.0|      92.0|
|Jerry Browning|             22.0|      92.0|
|   Andrea Frey|             10.0|      92.0|
+--------------+-----------------+----------+
only showing top 5 rows



In [11]:
#missing value
spark.sql(
    """
    SELECT
    StudentID,
    Name,
    StudyHoursPerWeek,
    FinalGrade
    FROM Student
    WHERE StudyHoursPerWeek IS NULL and FinalGrade IS NULL
    ORDER BY FinalGrade DESC
    """
).show()

+---------+------------------+-----------------+----------+
|StudentID|              Name|StudyHoursPerWeek|FinalGrade|
+---------+------------------+-----------------+----------+
|   2821.0|       Leslie King|             NULL|      NULL|
|   3440.0|Brandon Williamson|             NULL|      NULL|
|   1606.0|       Mary Benson|             NULL|      NULL|
+---------+------------------+-----------------+----------+



In [12]:
#Handling missing value

spark.sql(
    """
    SELECT
    StudentID,
    Name,
    StudyHoursPerWeek,
    FinalGrade
    FROM Student
    WHERE StudyHoursPerWeek IS NOT NULL and FinalGrade IS NOT NULL
    ORDER BY FinalGrade DESC
    """
).show()

+---------+-------------------+-----------------+----------+
|StudentID|               Name|StudyHoursPerWeek|FinalGrade|
+---------+-------------------+-----------------+----------+
|   1844.0|      Ronnie Taylor|             22.0|      92.0|
|   3994.0|        Angel Stout|             25.0|      92.0|
|   5836.0|      Anthony Smith|             30.0|      92.0|
|   8573.0|        James Smith|             12.0|      92.0|
|   1280.0|         Kelly Bell|             20.0|      92.0|
|   3900.0|        Andrea Frey|             10.0|      92.0|
|   6339.0|    Melissa Aguirre|              8.0|      92.0|
|   8461.0|      Kimberly Wood|             25.0|      92.0|
|   9907.0|Rebecca Mendoza DDS|             12.0|      92.0|
|   1166.0|     Tonya Gonzalez|             25.0|      92.0|
|   9880.0|   Darlene Fletcher|             12.0|      92.0|
|   2976.0|        Sarah Young|             20.0|      92.0|
|   8511.0|     Timothy Pruitt|             12.0|      92.0|
|   9165.0|    Angelica 

In [13]:
# mengganti Null menggunakan coelesce jadi nilai default

spark.sql(
    """
    SELECT
    StudentID,
    Name,
    COALESCE(FinalGrade,0) AS FinalGrade_Clean,
    COALESCE(StudyHoursPerWeek, 0) AS StudyHoursPerWeek_clean
    FROM Student
    """
).show(truncate = False)

+---------+-----------------+----------------+-----------------------+
|StudentID|Name             |FinalGrade_Clean|StudyHoursPerWeek_clean|
+---------+-----------------+----------------+-----------------------+
|1.0      |John             |80.0            |15.0                   |
|2.0      |Sarah            |87.0            |20.0                   |
|3.0      |Alex             |68.0            |10.0                   |
|4.0      |Michael          |92.0            |25.0                   |
|5.0      |Emma             |85.0            |18.0                   |
|6.0      |Olivia           |0.0             |30.0                   |
|7.0      |Daniel           |62.0            |8.0                    |
|8.0      |Sophia           |78.0            |17.0                   |
|9.0      |James            |72.0            |12.0                   |
|10.0     |Isabella         |88.0            |22.0                   |
|2824.0   |Shannon Harrison |87.0            |25.0                   |
|7912.

In [14]:
spark.sql(
    """
    SELECT
    StudentID,
    Name,
    COALESCE(FinalGrade,0) AS FinalGrade_Clean,
    COALESCE(StudyHoursPerWeek, 0) AS StudyHoursPerWeek_clean
    FROM Student
    """
).createOrReplaceTempView("student_clean")

In [15]:
spark.sql(
    """
    SELECT * FROM student_clean
    WHERE FinalGrade_Clean IS NULL
    """
).show() #sudah tidak ada angka NULL atau kosong

+---------+----+----------------+-----------------------+
|StudentID|Name|FinalGrade_Clean|StudyHoursPerWeek_clean|
+---------+----+----------------+-----------------------+
+---------+----+----------------+-----------------------+

